# Sparse PCA and Structured Sparsity for Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Implement Sparse PCA (SPCA) via alternating regression and identify features from sparse loadings
2. Compare features selected by SPCA loadings versus Lasso on a supervised task
3. Implement Group Lasso for datasets with known feature group structure
4. Visualise and interpret sparsity patterns in loading matrices and coefficient vectors

## Prerequisites
- Standard PCA (eigendecomposition, explained variance)
- Lasso regularisation (Module 4)
- Guide 02 of this module: *Structured Sparsity*

## Estimated Time: 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.linear_model import Lasso, LassoCV, ElasticNet
from sklearn.decomposition import SparsePCA, PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

## 1. Generating Structured High-Dimensional Data

We build a dataset with explicit group structure to make the structured sparsity methods interpretable:
- $p = 200$ features in 10 groups of 20 features each
- Features within a group share a common latent factor (high intra-group correlation)
- 3 groups are active (drive the outcome); the rest are noise
- Within each active group, 3–5 features carry the signal

In [ ]:
def generate_grouped_data(
    n: int = 300,
    n_groups: int = 10,
    group_size: int = 20,
    n_active_groups: int = 3,
    features_per_group: int = 4,
    intra_corr: float = 0.8,
    snr: float = 3.0,
    seed: int = 42
):
    """
    Generate grouped regression data with known active groups.

    Parameters
    ----------
    n : int
        Sample size.
    n_groups : int
        Total number of feature groups.
    group_size : int
        Number of features per group.
    n_active_groups : int
        Number of groups with non-zero effects.
    features_per_group : int
        Number of active features within each active group.
    intra_corr : float
        Within-group correlation (equicorrelation).
    snr : float
        Signal-to-noise ratio.
    seed : int

    Returns
    -------
    X : ndarray (n, p)
    y : ndarray (n,)
    groups : ndarray (p,)
        Group membership index for each feature.
    true_active : ndarray
        Indices of truly active features.
    active_groups : ndarray
        Indices of active groups.
    beta_true : ndarray (p,)
    """
    rng = np.random.default_rng(seed)
    p = n_groups * group_size

    # Build block-correlated design matrix
    X_list = []
    for g in range(n_groups):
        # Common factor within group
        common = rng.standard_normal(n)
        # Idiosyncratic variation
        idio = rng.standard_normal((n, group_size))
        # Equicorrelation: X_gj = sqrt(rho)*common + sqrt(1-rho)*idio_j
        block = np.sqrt(intra_corr) * common[:, None] + np.sqrt(1 - intra_corr) * idio
        X_list.append(block)

    X = np.hstack(X_list)  # (n, p)
    groups = np.repeat(np.arange(n_groups), group_size)

    # Active groups and features
    active_groups = np.arange(n_active_groups)  # first n_active_groups groups are active

    beta_true = np.zeros(p)
    true_active = []
    for g in active_groups:
        group_start = g * group_size
        # Select features_per_group features from this group
        active_within = rng.choice(group_size, size=features_per_group, replace=False)
        active_idx = group_start + active_within
        true_active.extend(active_idx.tolist())
        # Coefficients with varying magnitudes
        magnitudes = rng.uniform(0.5, 2.0, size=features_per_group)
        signs = rng.choice([-1, 1], size=features_per_group)
        beta_true[active_idx] = magnitudes * signs

    true_active = np.array(sorted(true_active))

    # Outcome
    signal = X @ beta_true
    noise_var = np.var(signal) / snr
    y = signal + rng.standard_normal(n) * np.sqrt(noise_var)

    # Standardise X
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, groups, true_active, active_groups, beta_true


X, y, groups, true_active, active_groups, beta_true = generate_grouped_data()
n, p = X.shape
n_groups = len(np.unique(groups))
group_size = p // n_groups

print(f"Dataset: n={n}, p={p}, {n_groups} groups of {group_size} features")
print(f"Active groups: {active_groups}")
print(f"Active features: {sorted(true_active)}")
print(f"Non-zero coefficients: {beta_true[true_active].round(2)}")

## 2. Standard PCA vs Sparse PCA

Standard PCA produces dense loadings — every feature contributes to every principal component. Sparse PCA constrains the loadings to be sparse, making components interpretable as "groups" of related features.

In [ ]:
# Standard PCA: dense loadings
n_components = 6
pca = PCA(n_components=n_components)
pca.fit(X)
pca_loadings = pca.components_.T  # (p, k)

print("Standard PCA:")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_.round(3)}")
print(f"  Non-zero loadings in PC1: {np.sum(np.abs(pca_loadings[:, 0]) > 1e-6)}/{p} (all non-zero by design)")

# Sparse PCA: sparse loadings via scikit-learn's implementation
# alpha controls sparsity: higher alpha → more zero loadings
spca = SparsePCA(
    n_components=n_components,
    alpha=1.5,          # L1 penalty on loadings
    max_iter=200,
    random_state=42,
    n_jobs=-1
)
spca.fit(X)
spca_loadings = spca.components_.T  # (p, k)

print("\nSparse PCA (alpha=1.5):")
for k in range(n_components):
    n_nonzero = np.sum(np.abs(spca_loadings[:, k]) > 1e-6)
    print(f"  Component {k+1}: {n_nonzero}/{p} non-zero loadings")

## 3. Visualising the Sparsity Pattern

The loading matrix heatmap reveals which features each component "focuses on". For well-structured data, we expect components to align with the known feature groups.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Custom colormap: blue (negative) - white (zero) - red (positive)
cmap = plt.cm.RdBu_r

# Standard PCA loadings
ax = axes[0]
im = ax.imshow(pca_loadings.T, cmap=cmap, aspect='auto',
               vmin=-np.abs(pca_loadings).max(), vmax=np.abs(pca_loadings).max())
ax.set_xlabel('Features (p=200)')
ax.set_ylabel('Principal Components')
ax.set_title('Standard PCA — Dense Loadings (every feature non-zero)')
ax.set_yticks(range(n_components))
ax.set_yticklabels([f'PC{k+1}' for k in range(n_components)])

# Add vertical lines to mark group boundaries
for g in range(1, n_groups):
    ax.axvline(g * group_size - 0.5, color='black', linewidth=0.5, alpha=0.5)
plt.colorbar(im, ax=ax, label='Loading value', shrink=0.8)

# Sparse PCA loadings
ax = axes[1]
spca_abs_max = max(np.abs(spca_loadings).max(), 1e-6)
im2 = ax.imshow(spca_loadings.T, cmap=cmap, aspect='auto',
                vmin=-spca_abs_max, vmax=spca_abs_max)
ax.set_xlabel('Features (p=200)')
ax.set_ylabel('Sparse Components')
ax.set_title('Sparse PCA — Sparse Loadings (most features zero per component)')
ax.set_yticks(range(n_components))
ax.set_yticklabels([f'SPC{k+1}' for k in range(n_components)])

for g in range(1, n_groups):
    ax.axvline(g * group_size - 0.5, color='black', linewidth=0.5, alpha=0.5)

# Highlight active groups
for g in active_groups:
    ax.axvspan(g * group_size - 0.5, (g+1) * group_size - 0.5,
               alpha=0.15, color='gold', label=f'Active group {g}' if g == 0 else '')

plt.colorbar(im2, ax=ax, label='Loading value', shrink=0.8)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.suptitle('PCA vs Sparse PCA: Loading Patterns', y=1.01, fontsize=13)
plt.show()

## 4. Feature Selection from Sparse PCA Loadings

We extract feature importance from the sparse loading matrix three ways:
1. **Union**: select any feature with at least one non-zero loading
2. **Max-loading**: rank by maximum absolute loading across components
3. **Variance-weighted**: weight by the variance explained per component

In [ ]:
def select_from_spca_loadings(
    loadings: np.ndarray,
    pca_obj,
    n_select: int,
    method: str = 'max'
) -> np.ndarray:
    """
    Select features from Sparse PCA loading matrix.

    Parameters
    ----------
    loadings : ndarray (p, k)
        Sparse loading matrix from SPCA.
    pca_obj : fitted PCA object
        Used to get explained variance ratios for weighting.
    n_select : int
        Number of features to select.
    method : str
        'union'  — any non-zero loading counts
        'max'    — rank by max |loading| across components
        'weighted' — rank by sum of (loading^2 * explained_var) across components

    Returns
    -------
    selected : ndarray
        Indices of selected features.
    scores : ndarray (p,)
        Feature importance scores.
    """
    if method == 'union':
        scores = (np.abs(loadings) > 1e-6).any(axis=1).astype(float)
    elif method == 'max':
        scores = np.abs(loadings).max(axis=1)
    elif method == 'weighted':
        # Approximate variance explained by each SPCA component
        # using the PCA explained variance as a proxy
        var_weights = pca_obj.explained_variance_ratio_
        scores = (loadings**2 @ var_weights)
    else:
        raise ValueError(f"Unknown method: {method}")

    selected = np.argsort(scores)[::-1][:n_select]
    return selected, scores


# Select top features using each method
n_select = len(true_active)  # select same number as ground truth

for method in ['max', 'weighted', 'union']:
    sel, scores = select_from_spca_loadings(spca_loadings, pca, n_select, method=method)
    tp = len(set(sel) & set(true_active))
    recall = tp / len(true_active)
    print(f"SPCA ({method:>8}): selected {len(sel)}, recall={recall:.0%}, "
          f"TP={tp}/{len(true_active)}")

# Get the 'max' selection for visualisation
spca_selected, spca_scores = select_from_spca_loadings(spca_loadings, pca, n_select, 'max')

## 5. Comparing SPCA Selection with Lasso

SPCA is unsupervised — it finds features that explain variance in $\mathbf{X}$. Lasso is supervised — it finds features that predict $y$. These need not agree when the most variable features are not the most predictive ones.

In [ ]:
# Lasso selection (supervised)
lasso = LassoCV(cv=5, max_iter=10000, n_jobs=-1)
lasso.fit(X, y)
lasso_selected = np.where(np.abs(lasso.coef_) > 1e-8)[0]

lasso_tp = len(set(lasso_selected) & set(true_active))
lasso_fp = len(set(lasso_selected) - set(true_active))
lasso_fn = len(set(true_active) - set(lasso_selected))

spca_tp = len(set(spca_selected) & set(true_active))
spca_fp = len(set(spca_selected) - set(true_active))
spca_fn = len(set(true_active) - set(spca_selected))

print("Feature Selection Comparison")
print("=" * 45)
print(f"{'Method':<15} {'TP':>5} {'FP':>5} {'FN':>5} {'Recall':>8} {'Precision':>10}")
print("-" * 45)
print(f"{'Lasso':<15} {lasso_tp:>5} {lasso_fp:>5} {lasso_fn:>5} "
      f"{lasso_tp/len(true_active):>8.0%} {lasso_tp/max(1,len(lasso_selected)):>10.0%}")
print(f"{'SPCA (max)':<15} {spca_tp:>5} {spca_fp:>5} {spca_fn:>5} "
      f"{spca_tp/len(true_active):>8.0%} {spca_tp/max(1,n_select):>10.0%}")

# Agreement between methods
agreement = set(lasso_selected) & set(spca_selected)
print(f"\nFeatures selected by both: {len(agreement)}")
print(f"True active in both: {len(agreement & set(true_active))}")

## 6. Group Lasso for Structured Selection

When we know the group structure, Group Lasso enforces group-level sparsity: either all features in a group are selected or none are. This matches our data-generating process where entire groups are active or inactive.

We implement a simple Group Lasso using the proximal gradient algorithm.

In [ ]:
def group_lasso_proximal(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    lam: float,
    max_iter: int = 2000,
    tol: float = 1e-6,
    step_size: float = None
) -> np.ndarray:
    """
    Group Lasso via proximal gradient descent (Yuan & Lin, 2006).

    Minimises:
        (1/2n) ||y - X beta||^2 + lam * sum_g sqrt(|G_g|) * ||beta_{G_g}||_2

    Parameters
    ----------
    X : ndarray (n, p)
    y : ndarray (n,)
    groups : ndarray (p,)
        Group index for each feature. Values in {0, 1, ..., G-1}.
    lam : float
        Penalty parameter.
    max_iter : int
    tol : float
        Convergence tolerance on coefficient change.
    step_size : float
        Gradient step size. Defaults to 1 / (Lipschitz constant of gradient).

    Returns
    -------
    beta : ndarray (p,)
        Group Lasso coefficient vector.
    """
    n, p = X.shape
    unique_groups = np.unique(groups)

    # Lipschitz constant of the gradient = max eigenvalue of X^T X / n
    if step_size is None:
        # Estimate Lipschitz constant via power iteration (faster than full SVD)
        v = np.random.randn(p)
        for _ in range(20):
            v = X.T @ (X @ v) / n
            norm_v = np.linalg.norm(v)
            if norm_v > 0:
                v = v / norm_v
        L = np.linalg.norm(X.T @ (X @ v) / n)
        step_size = 1.0 / (L + 1e-8)

    beta = np.zeros(p)

    for iteration in range(max_iter):
        beta_old = beta.copy()

        # Gradient step: grad = -X^T (y - X beta) / n
        residual = y - X @ beta
        grad = -X.T @ residual / n
        beta_half = beta - step_size * grad  # gradient step

        # Proximal step: group-wise block soft-thresholding
        beta = np.zeros(p)
        for g in unique_groups:
            g_mask = groups == g
            g_size = np.sum(g_mask)
            v_g = beta_half[g_mask]
            v_g_norm = np.linalg.norm(v_g)
            # Prox of ||.||_2: soft-threshold the entire group
            threshold = lam * np.sqrt(g_size) * step_size
            if v_g_norm > threshold:
                beta[g_mask] = v_g * (1 - threshold / v_g_norm)
            # else: beta[g_mask] = 0 (already zero from initialisation)

        # Check convergence
        if np.max(np.abs(beta - beta_old)) < tol:
            break

    return beta


# Fit Group Lasso across a grid of lambda values
lambda_grid = np.logspace(-2, 0.5, 30)
gl_results = []

print("Fitting Group Lasso across lambda grid...")
for lam in lambda_grid:
    beta_gl = group_lasso_proximal(X, y, groups, lam=lam)
    # Compute group norms
    group_norms = np.array([
        np.linalg.norm(beta_gl[groups == g]) for g in range(n_groups)
    ])
    selected_groups = np.where(group_norms > 1e-6)[0]
    selected_features = np.where(np.abs(beta_gl) > 1e-6)[0]
    tp_groups = len(set(selected_groups) & set(active_groups))
    gl_results.append({
        'lam': lam,
        'beta': beta_gl,
        'selected_groups': selected_groups,
        'tp_groups': tp_groups,
        'n_groups_selected': len(selected_groups),
        'selected_features': selected_features
    })

# Find best lambda (most true group recall with fewest false groups)
best = max(gl_results, key=lambda r: (
    r['tp_groups'], -r['n_groups_selected']
))
print(f"\nBest lambda: {best['lam']:.4f}")
print(f"Groups selected: {sorted(best['selected_groups'])}")
print(f"True active groups: {sorted(active_groups)}")
print(f"Group-level TP: {best['tp_groups']}/{len(active_groups)}")

## 7. Visualising the Group Lasso Solution Path

The solution path shows how group norms change as $\lambda$ varies. True active groups should maintain non-zero norms at higher $\lambda$ values than noise groups.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Group norm solution path
ax = axes[0]
group_norm_paths = np.zeros((len(lambda_grid), n_groups))
for i, res in enumerate(gl_results):
    for g in range(n_groups):
        group_norm_paths[i, g] = np.linalg.norm(res['beta'][groups == g])

for g in range(n_groups):
    is_active = g in active_groups
    color = 'red' if is_active else 'steelblue'
    alpha = 0.9 if is_active else 0.3
    lw = 2.5 if is_active else 0.8
    label = f'Group {g} (active)' if is_active else ('Noise groups' if g == n_groups-1 else None)
    ax.semilogx(lambda_grid, group_norm_paths[:, g], color=color, alpha=alpha, lw=lw, label=label)

ax.axvline(best['lam'], color='orange', linestyle='--', label=f"Best λ={best['lam']:.3f}")
ax.set_xlabel('Regularisation λ')
ax.set_ylabel('Group L2 norm ||β_g||₂')
ax.set_title('Group Lasso Solution Path')
ax.legend(fontsize=9)

# Panel 2: Coefficient comparison at best lambda
ax = axes[1]
beta_best = best['beta']

x_pos = np.arange(p)
colors_bar = ['red' if i in true_active else
              ('orange' if i in best['selected_features'] else 'steelblue')
              for i in range(p)]

# Plot with group separators
ax.bar(x_pos, beta_best, color=colors_bar, alpha=0.7, width=1.0)
ax.bar(x_pos[true_active], beta_true[true_active], color='darkred', alpha=0.5,
       width=1.0, label='True coefficients')

for g in range(1, n_groups):
    ax.axvline(g * group_size - 0.5, color='black', linewidth=0.5, alpha=0.5)
for g in active_groups:
    ax.axvspan(g * group_size - 0.5, (g+1) * group_size - 0.5,
               alpha=0.1, color='yellow')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Group Lasso + true active'),
    Patch(facecolor='orange', label='Group Lasso (false positive)'),
    Patch(facecolor='steelblue', label='Zeroed by Group Lasso'),
    Patch(facecolor='yellow', alpha=0.3, label='Active group region'),
]
ax.legend(handles=legend_elements, fontsize=8)
ax.set_xlabel('Feature index')
ax.set_ylabel('Coefficient value')
ax.set_title(f'Group Lasso Coefficients (λ={best["lam"]:.3f})')

plt.tight_layout()
plt.show()

## 8. Prediction Comparison: Lasso vs Group Lasso vs SPCA + Lasso

We compare predictive performance on a held-out test set to assess whether structured selection translates to better generalisation.

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Method 1: Standard Lasso
lasso_tr = LassoCV(cv=5, max_iter=10000)
lasso_tr.fit(X_train, y_train)
pred_lasso = lasso_tr.predict(X_test)
mse_lasso = mean_squared_error(y_test, pred_lasso)

# Method 2: Group Lasso (use best lambda found on full data — simplified for comparison)
beta_gl_train = group_lasso_proximal(X_train, y_train, groups, lam=best['lam'])
pred_gl = X_test @ beta_gl_train + np.mean(y_train)  # add mean back
mse_gl = mean_squared_error(y_test, pred_gl)

# Method 3: SPCA feature selection + OLS/Lasso
spca_tr = SparsePCA(n_components=6, alpha=1.5, max_iter=200, random_state=42)
spca_tr.fit(X_train)
sp_loadings_tr = spca_tr.components_.T
spca_sel_tr, _ = select_from_spca_loadings(sp_loadings_tr, pca, n_select, 'max')
lasso_spca = LassoCV(cv=5, max_iter=5000)
lasso_spca.fit(X_train[:, spca_sel_tr], y_train)
pred_spca = lasso_spca.predict(X_test[:, spca_sel_tr])
mse_spca = mean_squared_error(y_test, pred_spca)

# Oracle: OLS on true active features
from sklearn.linear_model import LinearRegression
oracle = LinearRegression()
oracle.fit(X_train[:, true_active], y_train)
pred_oracle = oracle.predict(X_test[:, true_active])
mse_oracle = mean_squared_error(y_test, pred_oracle)

print("Prediction Performance (test MSE)")
print("=" * 50)
print(f"{'Method':<25} {'Test MSE':>10} {'vs Oracle':>12}")
print("-" * 50)
print(f"{'Lasso':<25} {mse_lasso:>10.4f} {mse_lasso/mse_oracle:>11.2f}×")
print(f"{'Group Lasso':<25} {mse_gl:>10.4f} {mse_gl/mse_oracle:>11.2f}×")
print(f"{'SPCA + Lasso':<25} {mse_spca:>10.4f} {mse_spca/mse_oracle:>11.2f}×")
print(f"{'Oracle (true features)':<25} {mse_oracle:>10.4f} {'1.00×':>12}")

## 9. Sparsity Pattern Comparison Across Methods

A comprehensive visualisation showing which features each method selects, colour-coded by ground truth status.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

methods = {
    'True active': true_active,
    'Lasso': lasso_selected,
    'Group Lasso': best['selected_features'],
    'SPCA (max)': spca_selected,
}

# Binary selection matrix: rows = methods, cols = features
n_methods = len(methods)
selection_matrix = np.zeros((n_methods, p))
for row, (name, sel) in enumerate(methods.items()):
    selection_matrix[row, sel] = 1

# Colour-code: green=TP, red=FP, white=TN, blue=FN
colored_matrix = np.zeros((n_methods, p, 3))
for row in range(n_methods):
    for col in range(p):
        is_selected = selection_matrix[row, col] == 1
        is_true = col in set(true_active)
        if row == 0:  # Ground truth row
            colored_matrix[row, col] = [0.2, 0.6, 0.2]  # green
        elif is_selected and is_true:
            colored_matrix[row, col] = [0.2, 0.7, 0.2]  # TP: green
        elif is_selected and not is_true:
            colored_matrix[row, col] = [0.9, 0.2, 0.2]  # FP: red
        elif not is_selected and is_true:
            colored_matrix[row, col] = [0.1, 0.4, 0.9]  # FN: blue
        else:
            colored_matrix[row, col] = [0.95, 0.95, 0.95]  # TN: light grey

ax.imshow(colored_matrix, aspect='auto')
ax.set_yticks(range(n_methods))
ax.set_yticklabels(list(methods.keys()))
ax.set_xlabel('Feature index')
ax.set_title('Feature Selection Patterns (green=TP, red=FP, blue=FN, grey=TN)')

# Group boundary lines
for g in range(1, n_groups):
    ax.axvline(g * group_size - 0.5, color='black', linewidth=1.0, alpha=0.6)

# Active group shading
for g in active_groups:
    ax.axvspan(g * group_size - 0.5, (g+1) * group_size - 0.5,
               alpha=0.05, color='gold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#33b233', label='True positive (correct selection)'),
    Patch(facecolor='#e63333', label='False positive (incorrect selection)'),
    Patch(facecolor='#1a66e8', label='False negative (missed active feature)'),
    Patch(facecolor='#f2f2f2', label='True negative (correctly excluded)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## Summary

### Key Takeaways

1. **Standard PCA vs Sparse PCA**: Standard PCA gives dense loadings — every feature contributes to every component. Sparse PCA forces each component to focus on a small feature subset, making the components interpretable as latent feature groups.

2. **SPCA is unsupervised**: SPCA selects features that explain variance in $\mathbf{X}$, not in $y$. For prediction tasks, always follow SPCA feature discovery with a supervised selection step.

3. **Group Lasso enforces group sparsity**: When features have known group structure, Group Lasso selects entire groups in/out via block soft-thresholding. This is more appropriate than element-wise Lasso when domain knowledge specifies group structure.

4. **Structured penalties can outperform Lasso** when the group structure is correctly specified — they impose the right inductive bias and typically select fewer, more interpretable features.

5. **Visualise the sparsity pattern**: The loading heatmap and coefficient bar chart are essential diagnostics. If the sparse PCA components do not align with known groups, the sparsity parameter $\alpha$ needs tuning.

### What's Next

Notebook 03 (`03_post_selection_inference.ipynb`) addresses the statistical consequences of selection: how to construct valid confidence intervals after using the data to select features.

### Additional Resources

- **Zou, Hastie & Tibshirani (2006)**: SPCA — *JCGS* 15(2), 265–286
- **Yuan & Lin (2006)**: Group Lasso — *JRSS-B* 68(1), 49–67
- `sklearn.decomposition.SparsePCA`: scikit-learn's Sparse PCA implementation